# SIPTA — Ingesta y EDA: Seguridad y Convivencia (Cuadrantes MEBOG)
**Proyecto**: Sistema de Indicadores y Priorización Territorial y Alertas Tempranas (DataJam Bogotá)  
**Fase PDCO**: DEVELOPMENT | **Fase CRISP-DM**: Data Understanding / Data Preparation  
**Autoría**: **Persona C (Sofía Hidalgo — Ingesta & EDA)**  
**Objetivo**: Ingesta y análisis exploratorio (EDA) de los 599 cuadrantes policiales del MNVCC y cobertura distrital.  
**Datos de Entrada**: `data/raw/SEGURIDAD/*`  
**Datos de Salida**: `data/processed/SEGURIDAD/*`


## 1. Ingesta y Análisis Exploratorio de Seguridad Ciudadana



# SIPTA Notebook: Ingestión de datos — Dominio Seguridad
Este notebook documenta la prueba de lectura, exploración inicial de estructura y versionado de los datos crudos del dominio de Seguridad en el proyecto SIPTA.

### Objetivos
- Cargar y explorar el dataset de **Cuadrantes de Policía de Bogotá** desde `data/raw/SEGURIDAD/`.
- Validar las llaves territoriales distritales (`PCUIULOCAL` - Localidad / `PCUIUUPLAN` - UPZ).
- Evaluar completitud inicial (nulos, duplicados, dimensiones).
- Guardar copia de seguridad versionada para reproducibilidad.

In [ ]:
from pathlib import Path
import logging
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

for p in [Path('.').resolve(), Path('.').resolve().parent, Path('.').resolve().parent.parent]:
    if (p / 'src').exists():
        ROOT = p
        if str(ROOT) not in sys.path:
            sys.path.insert(0, str(ROOT))
        break
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Directorio Raw: {RAW_DIR}")
print(f"Directorio Processed: {PROCESSED_DIR}")


## 1. Función de Carga Reutilizable

In [ ]:
def load_raw_csv(filename: str, subcarpeta: str = "", separador: str = ';', encoding: str = 'latin1') -> pd.DataFrame:
    """Carga un dataset crudo en formato CSV con manejo de delimitador y codificación."""
    path = RAW_DIR / subcarpeta / filename if subcarpeta else RAW_DIR / filename
    assert path.exists(), f'No existe el archivo en la ruta: {path}'
    return pd.read_csv(path, sep=separador, low_memory=False, encoding=encoding)

def save_versioned_raw(df: pd.DataFrame, filename: str, subcarpeta: str = "", suffix: str = 'v1') -> Path:
    """Guarda versión inmutable en data/raw."""
    directorio = RAW_DIR / subcarpeta if subcarpeta else RAW_DIR
    directorio.mkdir(parents=True, exist_ok=True)
    output = directorio / f"{Path(filename).stem}_{suffix}.csv"
    df.to_csv(output, index=False, encoding='utf-8')
    logging.info(f"Copia versionada guardada en: {output}")
    return output

## 2. Ingesta: Cuadrantes de Policía de Bogotá

In [ ]:
# Carga de Cuadrantes de Policía
df_cuadrantes = load_raw_csv('Cuadrante de Policía. Bogotá D.C.csv', subcarpeta='SEGURIDAD', separador=';', encoding='latin1')

print(f"Dataset Cuadrantes cargado: {df_cuadrantes.shape[0]} filas, {df_cuadrantes.shape[1]} columnas.")

In [ ]:
print("=== 5 PRIMERAS FILAS: CUADRANTES DE POLICÍA ===")
cols_clave = [
    'properties/PCUNCUADRA',
    'properties/PCUNOMEST',
    'properties/PCUNOMCAI',
    'properties/PCUIULOCAL',
    'properties/PCUDESCRIP',
    'properties/PCUIUUPLAN'
]
display(df_cuadrantes[cols_clave].head())

In [ ]:
print("=== REVISIÓN TERRITORIAL: SEGURIDAD ===")
print("Códigos de localidades cubiertos (PCUIULOCAL):")
print(sorted(df_cuadrantes['properties/PCUIULOCAL'].dropna().astype(str).unique()))

print("\nEstaciones de Policía (PCUNOMEST):")
print(sorted(df_cuadrantes['properties/PCUNOMEST'].dropna().unique()))

print(f"\nTotal de cuadrantes únicos: {df_cuadrantes['properties/PCUNCUADRA'].nunique()}")

In [ ]:
print("=== ANÁLISIS DE NULOS EN VARIABLES CLAVE ===")
nulos_cuad = df_cuadrantes[cols_clave].isna().sum()
pct_cuad = (df_cuadrantes[cols_clave].isna().mean() * 100).round(2)
display(pd.DataFrame({'nulos': nulos_cuad, 'porcentaje (%)': pct_cuad}))

print(f"\nDuplicados exactos: {df_cuadrantes.duplicated().sum()}")

In [ ]:
# Guardar respaldo versionado
save_versioned_raw(df_cuadrantes, 'Cuadrante de Policía. Bogotá D.C.csv', subcarpeta='SEGURIDAD', suffix='20260731')

### Notas de Ingesta — Seguridad
- **Origen:** Policía Metropolitana de Bogotá (MEBOG) / Datos Abiertos Bogotá.
- **Ubicación:** `data/raw/SEGURIDAD/Cuadrante de Policía. Bogotá D.C.csv`.
- **Parámetros:** Separador `;` y codificación `latin1`.
- **Volumetría:** 599 cuadrantes de policía identificados.
- **Llaves territoriales:** `properties/PCUIULOCAL` (código de localidad del 1 al 19) y `properties/PCUIUUPLAN` (UPZ).
- **⚠ Pendiente de validar con datos:** Localidad 20 (Sumapaz) no registra cuadrantes urbanos tradicionales bajo este esquema; se validará su cobertura rural en la fase de integración.